In [1]:
import pandas as pd
import numpy as np
import random, hashlib, json, os
from datetime import datetime, timedelta

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_TRANSACTIONS = 50_000
FRAUD_RATE     = 0.02
N_CLIENTS      = 5_000

print("=" * 60)
print("  GÉNÉRATION DU DATASET FRAUDE BANCAIRE")
print("=" * 60)

# ── Villes marocaines avec coordonnées ────────────────────
MOROCCAN_CITIES = [
    "Casablanca", "Rabat", "Marrakech", "Fes", "Tanger",
    "Agadir", "Meknes", "Oujda", "El Jadida", "Safi",
    "Mohammedia", "Kenitra", "Tetouan", "Beni Mellal", "Khouribga"
]
CITY_COORDS = {
    "Casablanca":  (33.5731, -7.5898), "Rabat":      (34.0209, -6.8416),
    "Marrakech":   (31.6295, -7.9811), "Fes":        (34.0181, -5.0078),
    "Tanger":      (35.7595, -5.8340), "Agadir":     (30.4278, -9.5981),
    "Meknes":      (33.8935, -5.5547), "Oujda":      (34.6867, -1.9114),
    "El Jadida":   (33.2316, -8.5007), "Safi":       (32.2994, -9.2372),
    "Mohammedia":  (33.6861, -7.3830), "Kenitra":    (34.2610, -6.5802),
    "Tetouan":     (35.5785, -5.3684), "Beni Mellal":(32.3394, -6.3498),
    "Khouribga":   (32.8811, -6.9063),
}

FOREIGN_COORDS = [
    (48.8566, 2.3522),   # Paris
    (51.5074, -0.1278),  # London
    (6.5244, 3.3792),    # Lagos
    (25.2048, 55.2708),  # Dubai
    (40.4168, -3.7038),  # Madrid
    (50.8503, 4.3517),   # Brussels
    (41.0082, 28.9784),  # Istanbul
    (30.0444, 31.2357),  # Cairo
    (40.7128, -74.0060), # New York
    (47.3769, 8.5417),   # Zurich
]
FOREIGN_COUNTRIES = ["FR","GB","NG","AE","ES","BE","TR","EG","US","CH"]

TRANSACTION_TYPES = ["achat_tpe","achat_en_ligne","retrait_dab","virement","paiement_facture"]
DEVICE_TYPES      = ["mobile_app","web_browser","atm","tpe_physique","telephone"]
INCOME_SEGMENTS   = ["low","medium","high"]
CARD_TYPES        = ["debit","credit","prepaid"]


  GÉNÉRATION DU DATASET FRAUDE BANCAIRE


In [2]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi   = np.radians(lat2 - lat1)
    dlambda= np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def get_hour_risk(h):
    if 1 <= h <= 5:   return 3
    if 22 <= h <= 24 or h == 0: return 2
    if 6 <= h <= 8:   return 1
    return 0

In [3]:
# ── Profils clients ───────────────────────────────────────
print("\n[1/4] Génération des profils clients...")
clients = []
for i in range(N_CLIENTS):
    city = random.choice(MOROCCAN_CITIES)
    lat, lon = CITY_COORDS[city]
    clients.append({
        "client_id":          f"CLT{i:05d}",
        "city":               city,
        "home_lat":           round(lat + np.random.normal(0, 0.05), 4),
        "home_lon":           round(lon + np.random.normal(0, 0.05), 4),
        "age":                random.randint(18, 75),
        "avg_monthly_spend":  round(random.uniform(500, 15000), 2),
        "account_age_days":   random.randint(30, 3650),
        "income_segment":     random.choice(INCOME_SEGMENTS),
        "type_carte":         random.choice(CARD_TYPES),
    })
cdf = pd.DataFrame(clients).set_index("client_id")
print(f"   {N_CLIENTS} profils générés.")

# ── Génération des transactions ───────────────────────────
print("\n[2/4] Génération des transactions...")

N_FRAUD  = int(N_TRANSACTIONS * FRAUD_RATE)
N_LEGIT  = N_TRANSACTIONS - N_FRAUD
rows     = []
base_dt  = datetime(2024, 1, 1)
client_last = {}   # client_id -> (ts, lat, lon)
client_times= {}   # client_id -> [recent timestamps]



[1/4] Génération des profils clients...
   5000 profils générés.

[2/4] Génération des transactions...


In [4]:
# ── LÉGITIMES ─────────────────────────────────────────────
for _ in range(N_LEGIT):
    cid    = f"CLT{random.randint(0, N_CLIENTS-1):05d}"
    c      = cdf.loc[cid]
    days   = random.randint(0, 364)
    hour   = int(np.clip(np.random.normal(13, 4), 6, 22))
    minute = random.randint(0, 59)
    ts     = base_dt + timedelta(days=days, hours=hour, minutes=minute)

    tx_lat = c["home_lat"] + np.random.normal(0, 0.08)
    tx_lon = c["home_lon"] + np.random.normal(0, 0.08)

    base   = c["avg_monthly_spend"] / 30
    montant= abs(np.random.normal(base, base * 0.3))
    montant= round(min(montant, c["avg_monthly_spend"] * 0.5), 2)

    delta_km, delta_min = 0.0, 9999.0
    if cid in client_last:
        lts, llat, llon = client_last[cid]
        delta_km  = haversine_km(llat, llon, tx_lat, tx_lon)
        delta_min = (ts - lts).total_seconds() / 60

    recent = [t for t in client_times.get(cid, []) if (ts - t).total_seconds() < 3600]
    nb_tx_1h = len(recent)
    ratio_m  = round(montant / (base + 1e-9), 3)

    rows.append({
        "tx_id":            f"TX{len(rows):08d}",
        "client_id":         cid,
        "timestamp":         ts.strftime("%Y-%m-%d %H:%M:%S"),
        "heure":             hour,
        "jour_semaine":      ts.weekday(),
        "est_weekend":       int(ts.weekday() >= 5),
        "montant_mad":       montant,
        "type_transaction":  random.choice(TRANSACTION_TYPES),
        "pays_transaction":  "MA",
        "est_etranger":      0,
        "tx_lat":            round(tx_lat, 4),
        "tx_lon":            round(tx_lon, 4),
        "delta_km":          round(delta_km, 2),
        "delta_min_last_tx": round(delta_min, 1),
        "nb_tx_1h":          nb_tx_1h,
        "device_type":       random.choice(DEVICE_TYPES),
        "est_nouveau_device":int(random.random() < 0.05),
        "age_client":        int(c["age"]),
        "segment_revenu":    c["income_segment"],
        "type_carte":        c["type_carte"],
        "age_compte_jours":  int(c["account_age_days"]),
        "ratio_montant_moy": ratio_m,
        "risque_horaire":    get_hour_risk(hour),
        "fraude":            0,
        "motif_fraude":      "none",
    })
    client_last[cid]  = (ts, tx_lat, tx_lon)
    client_times[cid] = recent + [ts]



In [5]:
# ── FRAUDULEUSES ──────────────────────────────────────────
PATTERNS = ["clonage_carte","phishing","usurpation_identite","fraude_cni","test_micro_montants"]

for _ in range(N_FRAUD):
    cid     = f"CLT{random.randint(0, N_CLIENTS-1):05d}"
    c       = cdf.loc[cid]
    pattern = random.choice(PATTERNS)
    days    = random.randint(0, 364)
    hour    = random.randint(0, 5) if random.random() < 0.6 else random.choice(list(range(0,24)))
    minute  = random.randint(0, 59)
    ts      = base_dt + timedelta(days=days, hours=hour, minutes=minute)
    base    = c["avg_monthly_spend"] / 30

    if pattern == "clonage_carte":
        fi      = random.randint(0, len(FOREIGN_COORDS)-1)
        tx_lat, tx_lon = FOREIGN_COORDS[fi]
        country = FOREIGN_COUNTRIES[fi]
        delta_km  = haversine_km(c["home_lat"], c["home_lon"], tx_lat, tx_lon)
        delta_min = random.uniform(20, 180)
        montant   = round(random.uniform(1000, 15000), 2)
        nb_tx_1h  = random.randint(1, 3)
        new_dev   = random.random() < 0.8
        tx_type   = random.choice(["achat_tpe","retrait_dab"])

    elif pattern == "phishing":
        tx_lat = c["home_lat"] + np.random.normal(0, 0.3)
        tx_lon = c["home_lon"] + np.random.normal(0, 0.3)
        country   = "MA"
        delta_km  = random.uniform(5, 50)
        delta_min = random.uniform(5, 60)
        montant   = round(random.uniform(2000, 20000), 2)
        nb_tx_1h  = random.randint(0, 2)
        new_dev   = True
        tx_type   = "achat_en_ligne"

    elif pattern == "usurpation_identite":
        tx_lat = c["home_lat"] + np.random.normal(0, 0.1)
        tx_lon = c["home_lon"] + np.random.normal(0, 0.1)
        country   = random.choice(["MA","FR","ES"])
        delta_km  = random.uniform(1, 20)
        delta_min = random.uniform(1, 10)
        montant   = round(random.uniform(200, 3000), 2)
        nb_tx_1h  = random.randint(5, 15)
        new_dev   = random.random() < 0.6
        tx_type   = random.choice(TRANSACTION_TYPES)

    elif pattern == "fraude_cni":
        fi      = random.randint(0, 4)
        tx_lat, tx_lon = FOREIGN_COORDS[fi]
        country = FOREIGN_COUNTRIES[fi]
        delta_km  = haversine_km(c["home_lat"], c["home_lon"], tx_lat, tx_lon)
        delta_min = random.uniform(30, 120)
        montant   = round(random.uniform(500, 8000), 2)
        nb_tx_1h  = random.randint(2, 6)
        new_dev   = True
        tx_type   = random.choice(["achat_tpe","achat_en_ligne"])

    else:  # test_micro_montants
        tx_lat = c["home_lat"] + np.random.normal(0, 0.05)
        tx_lon = c["home_lon"] + np.random.normal(0, 0.05)
        country   = "MA"
        delta_km  = random.uniform(0, 5)
        delta_min = random.uniform(1, 5)
        montant   = round(random.uniform(1, 50), 2)
        nb_tx_1h  = random.randint(8, 20)
        new_dev   = random.random() < 0.4
        tx_type   = "achat_en_ligne"

    ratio_m = round(montant / (base + 1e-9), 3)
    rows.append({
        "tx_id":            f"TX{len(rows):08d}",
        "client_id":         cid,
        "timestamp":         ts.strftime("%Y-%m-%d %H:%M:%S"),
        "heure":             hour,
        "jour_semaine":      ts.weekday(),
        "est_weekend":       int(ts.weekday() >= 5),
        "montant_mad":       montant,
        "type_transaction":  tx_type,
        "pays_transaction":  country,
        "est_etranger":      int(country != "MA"),
        "tx_lat":            round(tx_lat, 4),
        "tx_lon":            round(tx_lon, 4),
        "delta_km":          round(delta_km, 2),
        "delta_min_last_tx": round(delta_min, 1),
        "nb_tx_1h":          nb_tx_1h,
        "device_type":       random.choice(DEVICE_TYPES),
        "est_nouveau_device":int(new_dev),
        "age_client":        int(c["age"]),
        "segment_revenu":    c["income_segment"],
        "type_carte":        c["type_carte"],
        "age_compte_jours":  int(c["account_age_days"]),
        "ratio_montant_moy": ratio_m,
        "risque_horaire":    get_hour_risk(hour),
        "fraude":            1,
        "motif_fraude":      pattern,
    })


In [6]:
# ── Sauvegarde ────────────────────────────────────────────
print("\n[3/4] Assemblage et sauvegarde...")
os.makedirs("data", exist_ok=True)
df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
# Bruit réaliste — 0.5% seulement (pas 3%)
noise_mask = np.random.random(len(df)) < 0.005
df.loc[noise_mask, "fraude"] = 1 - df.loc[noise_mask, "fraude"]

# Voyageurs légitimes — 60 seulement (pas 200)
n_travelers = 60
traveler_idx = df[df["fraude"] == 0].sample(n_travelers, random_state=42).index
df.loc[traveler_idx, "delta_km"] = np.random.uniform(1000, 4000, n_travelers)
df.loc[traveler_idx, "est_etranger"] = 1
df.to_csv("data/transactions_bancaires.csv", index=False)

sha256 = hashlib.sha256(open("data/transactions_bancaires.csv","rb").read()).hexdigest()

metadata = {
    "dataset_name":   "transactions_bancaires",
    "version":        "1.0.0",
    "n_transactions": len(df),
    "n_fraud":        int(df["fraude"].sum()),
    "n_legit":        int((df["fraude"]==0).sum()),
    "fraud_rate_pct": round(df["fraude"].mean()*100, 2),
    "n_clients":      N_CLIENTS,
    "sha256":         sha256,
    "features":       [c for c in df.columns if c not in ["tx_id","client_id","motif_fraude","fraude"]],
    "fraud_patterns": df[df["fraude"]==1]["motif_fraude"].value_counts().to_dict(),
    "generated_at":   datetime.now().isoformat(),
}
with open("data/dataset_metadata.json","w",encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("\n[4/4] Statistiques :")
print(f"   Transactions totales  : {len(df):,}")
print(f"   Légitimes             : {(df['fraude']==0).sum():,}  ({(df['fraude']==0).mean()*100:.1f}%)")
print(f"   Frauduleuses          : {df['fraude'].sum():,}  ({df['fraude'].mean()*100:.1f}%)")
print(f"   Features métier       : {len(metadata['features'])}")
print(f"   Hash SHA-256          : {sha256[:40]}...")
print(f"\n   Répartition des fraudes :")
for p, n in metadata["fraud_patterns"].items():
    print(f"     {p:<28} : {n:>4}")
print(f"\n   Fichiers : data/transactions_bancaires.csv")
print(f"              data/dataset_metadata.json")
print("\n" + "="*60)
print("  DATASET PRÊT — hash à enregistrer sur blockchain")
print("="*60)



[3/4] Assemblage et sauvegarde...

[4/4] Statistiques :
   Transactions totales  : 50,000
   Légitimes             : 48,760  (97.5%)
   Frauduleuses          : 1,240  (2.5%)
   Features métier       : 21
   Hash SHA-256          : 88fd9f20436ef10616b66e4e44acd793e25f265e...

   Répartition des fraudes :
     none                         :  247
     usurpation_identite          :  212
     phishing                     :  204
     clonage_carte                :  197
     fraude_cni                   :  194
     test_micro_montants          :  186

   Fichiers : data/transactions_bancaires.csv
              data/dataset_metadata.json

  DATASET PRÊT — hash à enregistrer sur blockchain
